In [ ]:
!pip install langchain-qdrant langchain-ollama fastembed

In [1]:
# ============================================================
# 0. SETUP + CONFIG
# ============================================================
# Chạy 1 lần từ trên xuống dưới.
# Yêu cầu: docling, transformers, pandas

import json
import os
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# ---------------- Config ----------------
PDF_PATH = Path("Z5UM_EU(En)04.pdf")

OUTPUT_DIR = Path("output")
CHUNKS_DIR = OUTPUT_DIR / "chunks"
IMAGES_DIR = OUTPUT_DIR / "images"
TABLES_DIR = OUTPUT_DIR / "tables"

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 512

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = PDF_PATH.name
CREATED_AT = datetime.now().astimezone().isoformat(timespec="seconds")

print(f"PDF: {PDF_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Source: {SOURCE_FILE}")
print(f"Created: {CREATED_AT}")


PDF: Z5UM_EU(En)04.pdf
Output: output
Source: Z5UM_EU(En)04.pdf
Created: 2026-10-01T00:46:58+07:00


In [3]:
# ============================================================
# 1. PDF PIPELINE + CONVERT
# ============================================================
# generate_picture_images=True : bắt buộc để lấy PictureItem
# generate_page_images=False   : nhẹ hơn, chỉ bật khi cần debug/render full trang

from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_page_images = False
pipeline_options.images_scale = 2.0
# pipeline_options.do_picture_description = True
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print(f"Processing: {PDF_PATH}")
result = converter.convert(PDF_PATH)
doc = result.document
print("PDF parsed successfully.")


2026-10-01 00:48:32,626 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-10-01 00:48:32,639 - INFO - Going to convert document batch...
2026-10-01 00:48:32,640 - INFO - Initializing pipeline for StandardPdfPipeline with options hash cdf150a01134d559192924d442aefd97
2026-10-01 00:48:32,693 - INFO - Loading plugin 'docling_defaults'
2026-10-01 00:48:32,760 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-10-01 00:48:32,761 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']


Processing: Z5UM_EU(En)04.pdf


2026-10-01 00:48:35,083 - INFO - Accelerator device: 'cpu'
[INFO] 2026-10-01 00:48:35,110 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-10-01 00:48:35,203 [RapidOCR] download_file.py:60: File exists and is valid: D:\anaconda3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-10-01 00:48:35,204 [RapidOCR] main.py:53: Using D:\anaconda3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-10-01 00:48:35,574 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-10-01 00:48:35,600 [RapidOCR] download_file.py:60: File exists and is valid: D:\anaconda3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-10-01 00:48:35,600 [RapidOCR] main.py:53: Using D:\anaconda3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-10-01 00:48:35,697 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-10-01 00:48:35,776 [RapidOCR] download_file.py:60: File exists

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-10-01 00:48:37,772 - INFO - Loading plugin 'docling_defaults'
2026-10-01 00:48:37,784 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-10-01 00:48:37,785 - INFO - Registered table structure engines: ['docling_tableformer']
2026-10-01 00:48:38,036 - INFO - HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-10-01 00:48:38,295 - INFO - Accelerator device: 'cpu'
2026-10-01 00:48:39,094 - INFO - Processing document Z5UM_EU(En)04.pdf
[INFO] 2026-10-01 00:48:43,667 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.4.0/resources/fonts/FZYTK.TTF
[INFO] 2026-10-01 00:48:45,970 [RapidOCR] download_file.py:82: Download size: 3.09MB
[INFO] 2026-10-01 00:49:02,641 [RapidOCR] download_file.py:95: Successfully saved to: D:\anaconda3\Lib\site-packages\rapidocr\models\FZYTK.TTF
2026

PDF parsed successfully.


In [4]:
# ============================================================
# 2. EXTRACT IMAGES -> output/images/*.png + images_metadata.json
# ============================================================

from docling_core.types.doc import PictureItem

print("Extracting images...")

picture_count = 0
image_metadata = []

for element, level in doc.iterate_items():
    if not isinstance(element, PictureItem):
        continue

    picture_count += 1
    image_id = f"picture_{picture_count:04d}"
    image_path = IMAGES_DIR / f"{image_id}.png"

    try:
        image = element.get_image(doc)
        image.save(image_path, "PNG")

        try:
            doc_item_ref = str(element.self_ref)
        except Exception:
            doc_item_ref = None

        page_no = None
        try:
            if element.prov:
                page_no = element.prov[0].page_no
        except Exception:
            pass

        image_metadata.append({
            "image_id": image_id,
            "file": (image_path.relative_to(OUTPUT_DIR)).as_posix(),  # images/picture_XXXX.png
            "source_file": SOURCE_FILE,
            "created_at": CREATED_AT,
            "doc_item_ref": doc_item_ref,
            "page_no": page_no,
            "width": image.width,
            "height": image.height,
            "level": level,
        })
        print(f"Image {picture_count}: {image_path}")
    except Exception as e:
        print(f"Failed image {picture_count}: {e}")

(OUTPUT_DIR / "images_metadata.json").write_text(
    json.dumps(image_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\nExtracted {picture_count} images -> {OUTPUT_DIR / 'images_metadata.json'}")


Extracting images...
Image 1: output\images\picture_0001.png
Image 2: output\images\picture_0002.png
Image 3: output\images\picture_0003.png
Image 4: output\images\picture_0004.png
Image 5: output\images\picture_0005.png
Image 6: output\images\picture_0006.png
Image 7: output\images\picture_0007.png
Image 8: output\images\picture_0008.png
Image 9: output\images\picture_0009.png
Image 10: output\images\picture_0010.png
Image 11: output\images\picture_0011.png
Image 12: output\images\picture_0012.png
Image 13: output\images\picture_0013.png
Image 14: output\images\picture_0014.png
Image 15: output\images\picture_0015.png
Image 16: output\images\picture_0016.png
Image 17: output\images\picture_0017.png
Image 18: output\images\picture_0018.png
Image 19: output\images\picture_0019.png
Image 20: output\images\picture_0020.png
Image 21: output\images\picture_0021.png
Image 22: output\images\picture_0022.png
Image 23: output\images\picture_0023.png
Image 24: output\images\picture_0024.png
Imag

In [5]:
# ============================================================
# 3. EXTRACT TABLES -> output/tables/*.{csv,html} + tables_metadata.json
# ============================================================

from docling_core.types.doc import TableItem

print("Extracting tables...")

table_count = 0
table_metadata = []

for element, level in doc.iterate_items():
    if not isinstance(element, TableItem):
        continue

    table_count += 1
    table_id = f"table_{table_count:04d}"
    csv_path = TABLES_DIR / f"{table_id}.csv"
    html_path = TABLES_DIR / f"{table_id}.html"

    try:
        doc_item_ref = str(element.self_ref)
    except Exception:
        doc_item_ref = None

    page_no = None
    try:
        if element.prov:
            page_no = element.prov[0].page_no
    except Exception:
        pass

    df = None
    try:
        df = element.export_to_dataframe(doc=doc)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    except Exception as e:
        print(f"Table {table_id} CSV failed: {e}")

    try:
        html = element.export_to_html(doc=doc)
        html_path.write_text(html, encoding="utf-8")
    except Exception as e:
        print(f"Table {table_id} HTML failed: {e}")

    table_metadata.append({
        "table_id": table_id,
        "source_file": SOURCE_FILE,
        "created_at": CREATED_AT,
        "doc_item_ref": doc_item_ref,
        "page_no": page_no,
        "csv": (csv_path.relative_to(OUTPUT_DIR)).as_posix() if csv_path.exists() else None,
        "html": (html_path.relative_to(OUTPUT_DIR)).as_posix() if html_path.exists() else None,
        "rows": len(df) if df is not None else None,
        "columns": len(df.columns) if df is not None else None,
        "column_names": [str(c) for c in df.columns] if df is not None else [],
        "level": level,
    })
    print(f"Table {table_count}: {table_id} (page={page_no})")

(OUTPUT_DIR / "tables_metadata.json").write_text(
    json.dumps(table_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\nExtracted {table_count} tables -> {OUTPUT_DIR / 'tables_metadata.json'}")


Extracting tables...
Table 1: table_0001 (page=1)
Table 2: table_0002 (page=6)
Table 3: table_0003 (page=7)
Table 4: table_0004 (page=8)
Table 5: table_0005 (page=9)
Table 6: table_0006 (page=10)
Table 7: table_0007 (page=11)
Table 8: table_0008 (page=13)
Table 9: table_0009 (page=14)
Table 10: table_0010 (page=42)
Table 11: table_0011 (page=43)
Table 12: table_0012 (page=44)
Table 13: table_0013 (page=45)
Table 14: table_0014 (page=46)
Table 15: table_0015 (page=47)
Table 16: table_0016 (page=48)
Table 17: table_0017 (page=49)
Table 18: table_0018 (page=50)
Table 19: table_0019 (page=51)
Table 20: table_0020 (page=52)
Table 21: table_0021 (page=53)
Table 22: table_0022 (page=54)
Table 23: table_0023 (page=54)
Table 24: table_0024 (page=56)
Table 25: table_0025 (page=70)
Table 26: table_0026 (page=72)
Table 27: table_0027 (page=73)
Table 28: table_0028 (page=75)

Extracted 28 tables -> output\tables_metadata.json


In [6]:
# ============================================================
# 4. HYBRID CHUNK
# ============================================================

from transformers import AutoTokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

hf_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

tokenizer = HuggingFaceTokenizer(
    tokenizer=hf_tokenizer,
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    repeat_table_header=True,
    merge_peers=True,
)

print("Chunking document...")
chunks = list(chunker.chunk(dl_doc=doc))
print(f"Total chunks: {len(chunks)}")


2026-10-01 00:50:10,617 - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-01 00:50:10,901 - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-01 00:50:11,151 - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1663 > 512). Running this sequence through the model will result in indexing errors


Chunking document...
Total chunks: 123


In [7]:
# ============================================================
# 5. BUILD CHUNKS + PARENT-CHILD STRUCTURE
#    + GAN ANH/BANG VAO METADATA
# ============================================================

from collections import defaultdict
from docling_core.types.doc import PictureItem

# ============================================================
# 5a. Asset maps (exact match)
# ============================================================

image_map = {
    m["doc_item_ref"]: m
    for m in image_metadata
    if m.get("doc_item_ref")
}

table_map = {
    m["doc_item_ref"]: m
    for m in table_metadata
    if m.get("doc_item_ref")
}

print(
    f"Images in map: {len(image_map)} | "
    f"Tables in map: {len(table_map)}"
)


# ============================================================
# 5b. Helpers
# ============================================================

def get_chunk_page_boxes(chunk):
    """
    Lấy bounding box tổng của các doc_items thuộc chunk,
    group theo page.
    """
    page_boxes = defaultdict(list)

    for item in chunk.meta.doc_items:
        for prov in getattr(item, "prov", []) or []:

            page_no = getattr(prov, "page_no", None)

            if page_no is None:
                continue

            bbox = getattr(prov, "bbox", None)

            if bbox is not None:
                page_boxes[page_no].append(bbox)

    out = {}

    for page_no, boxes in page_boxes.items():

        out[page_no] = {
            "left": min(b.l for b in boxes),
            "right": max(b.r for b in boxes),
            "bottom": min(b.b for b in boxes),
            "top": max(b.t for b in boxes),
        }

    return out


def vertical_distance(picture_bbox, chunk_box):
    """
    Khoảng cách theo chiều dọc giữa picture và chunk.
    """

    pt = picture_bbox.t
    pb = picture_bbox.b

    ct = chunk_box["top"]
    cb = chunk_box["bottom"]

    # picture nằm trong vùng chunk
    if pb <= ct and pt >= cb:
        return 0.0

    # picture nằm phía trên chunk
    if pb > ct:
        return pb - ct

    # picture nằm phía dưới chunk
    return cb - pt


def normalize_headings(headings):
    """
    Chuẩn hóa headings của HybridChunker thành list[str].
    """

    if not headings:
        return []

    result = []

    for heading in headings:

        if heading is None:
            continue

        # trường hợp heading đã là string
        if isinstance(heading, str):
            value = heading.strip()

        else:
            # một số version Docling có object
            value = str(heading).strip()

        if value:
            result.append(value)

    return result


def make_section_path(headings):
    """
    Convert:
        ["PHOTOSHOOTING MENU", "Picture Controls"]

    thành:
        PHOTOSHOOTING MENU > Picture Controls
    """

    if not headings:
        return ""

    return " > ".join(headings)


def make_parent_id(source_file, section_path):
    """
    Parent ID ổn định theo source + section.
    """

    import hashlib

    raw = f"{source_file}::{section_path}"

    digest = hashlib.md5(
        raw.encode("utf-8")
    ).hexdigest()[:12]

    return f"parent_{digest}"


# ============================================================
# 5c. Precompute chunk boxes
# ============================================================

chunk_boxes_list = [
    get_chunk_page_boxes(c)
    for c in chunks
]


# ============================================================
# 5d. Picture index + nearest chunk
# ============================================================

pictures_by_page = defaultdict(list)

for element, level in doc.iterate_items():

    if not isinstance(element, PictureItem):
        continue

    ref = str(element.self_ref)

    for prov in element.prov or []:

        page_no = getattr(prov, "page_no", None)
        bbox = getattr(prov, "bbox", None)

        if page_no is None or bbox is None:
            continue

        pictures_by_page[page_no].append(
            {
                "picture_ref": ref,
                "bbox": bbox,
            }
        )


picture_to_chunk = {}


for page, pics in pictures_by_page.items():

    for pic in pics:

        best = None
        best_d = float("inf")

        for idx, boxes in enumerate(chunk_boxes_list):

            if page not in boxes:
                continue

            d = vertical_distance(
                pic["bbox"],
                boxes[page],
            )

            if d < best_d:
                best_d = d
                best = idx

        if best is not None:

            prev = picture_to_chunk.get(
                pic["picture_ref"]
            )

            if (
                prev is None
                or best_d < prev["distance"]
            ):
                picture_to_chunk[
                    pic["picture_ref"]
                ] = {
                    "chunk_index": best,
                    "distance": best_d,
                }


bbox_refs_by_chunk = defaultdict(list)

for ref, mapping in picture_to_chunk.items():

    bbox_refs_by_chunk[
        mapping["chunk_index"]
    ].append(ref)


print(
    f"Pictures indexed: "
    f"{sum(len(v) for v in pictures_by_page.values())} | "
    f"mapped: {len(picture_to_chunk)}"
)


# ============================================================
# 5e. FIRST PASS
#     Build raw chunk information
# ============================================================

raw_chunks = []

last_headings = []

for index, chunk in enumerate(chunks):

    chunk_id = f"chunk_{index:04d}"

    chunk_path = (
        CHUNKS_DIR / f"{chunk_id}.md"
    )

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    text = chunker.contextualize(
        chunk=chunk
    )

    num_tokens = len(
        hf_tokenizer.encode(
            text,
            add_special_tokens=False,
        )
    )

    # --------------------------------------------------------
    # Headings từ HybridChunker
    # --------------------------------------------------------

    raw_headings = normalize_headings(
        getattr(chunk.meta, "headings", None)
    )

    # Nếu chunk hiện tại không có heading,
    # giữ section trước đó.
    if raw_headings:

        current_headings = raw_headings
        last_headings = raw_headings

    else:

        current_headings = last_headings

    section_path = make_section_path(
        current_headings
    )

    # --------------------------------------------------------
    # Parent
    # --------------------------------------------------------

    if section_path:

        parent_id = make_parent_id(
            SOURCE_FILE,
            section_path,
        )

        parent_title = (
            current_headings[-1]
            if current_headings
            else None
        )

    else:

        # Chunk không thuộc section nào
        parent_id = make_parent_id(
            SOURCE_FILE,
            "__ROOT__",
        )

        parent_title = "__ROOT__"

    # --------------------------------------------------------
    # Doc items
    # --------------------------------------------------------

    doc_items = [
        {
            "ref": str(item.self_ref),
            "label": str(item.label),
        }
        for item in chunk.meta.doc_items
    ]

    raw_chunks.append(
        {
            "index": index,
            "chunk": chunk,
            "chunk_id": chunk_id,
            "chunk_path": chunk_path,
            "text": text,
            "num_tokens": num_tokens,

            # hierarchy
            "headings": current_headings,
            "section_path": section_path,
            "parent_id": parent_id,
            "parent_title": parent_title,

            "doc_items": doc_items,
        }
    )


# ============================================================
# 5f. GROUP CHILD CHUNKS BY PARENT
# ============================================================

parent_groups = defaultdict(list)

for item in raw_chunks:

    parent_groups[
        item["parent_id"]
    ].append(
        item["index"]
    )


# ============================================================
# 5g. BUILD + SAVE FINAL CHUNKS
# ============================================================

chunk_metadata = []


for item in raw_chunks:

    index = item["index"]

    chunk_id = item["chunk_id"]

    chunk_path = item["chunk_path"]

    text = item["text"]

    # --------------------------------------------------------
    # Parent information
    # --------------------------------------------------------

    parent_id = item["parent_id"]

    parent_title = item["parent_title"]

    section_path = item["section_path"]

    parent_chunk_indices = (
        parent_groups[parent_id]
    )

    parent_chunk_ids = [
        f"chunk_{i:04d}"
        for i in parent_chunk_indices
    ]

    # --------------------------------------------------------
    # Exact images / tables
    # --------------------------------------------------------

    exact_images = []
    exact_tables = []

    exact_image_refs = set()

    for doc_item in item["chunk"].meta.doc_items:

        ref = str(doc_item.self_ref)

        # -------------------------------
        # Exact image
        # -------------------------------

        if (
            ref in image_map
            and ref not in exact_image_refs
        ):

            m = image_map[ref]

            exact_images.append(
                {
                    "image_id": m["image_id"],
                    "file": m["file"],
                    "page_no": m["page_no"],
                    "doc_item_ref": m["doc_item_ref"],
                    "match": "exact",
                }
            )

            exact_image_refs.add(ref)

        # -------------------------------
        # Exact table
        # -------------------------------

        if ref in table_map:

            m = table_map[ref]

            exact_tables.append(
                {
                    "table_id": m["table_id"],
                    "csv": m["csv"],
                    "html": m["html"],
                    "page_no": m["page_no"],
                    "doc_item_ref": m["doc_item_ref"],
                }
            )

    # --------------------------------------------------------
    # BBOX images
    # --------------------------------------------------------

    bbox_images = []

    for ref in bbox_refs_by_chunk.get(
        index,
        []
    ):

        # đã exact -> không duplicate
        if ref in exact_image_refs:
            continue

        m = image_map.get(ref)

        if m is None:
            continue

        bbox_images.append(
            {
                "image_id": m["image_id"],
                "file": m["file"],
                "page_no": m["page_no"],
                "doc_item_ref": m["doc_item_ref"],
                "match": "bbox",
            }
        )

    images = (
        exact_images
        + bbox_images
    )

    tables = exact_tables

    # ========================================================
    # FINAL METADATA
    # ========================================================

    metadata = {

        # --------------------------------------------
        # Basic
        # --------------------------------------------

        "chunk_id": chunk_id,

        "chunk_index": index,

        "source_file": SOURCE_FILE,

        "num_tokens": item["num_tokens"],

        # --------------------------------------------
        # Parent / Child
        # --------------------------------------------

        "parent_id": parent_id,

        "parent_title": parent_title,

        "section_path": section_path,

        "headings": item["headings"],

        "parent_chunk_indices": (
            parent_chunk_indices
        ),

        "parent_chunk_ids": (
            parent_chunk_ids
        ),

        # --------------------------------------------
        # Docling grounding
        # --------------------------------------------

        "doc_items": item["doc_items"],

        # --------------------------------------------
        # Assets
        # --------------------------------------------

        "images": images,

        "tables": tables,

        # --------------------------------------------
        # File
        # --------------------------------------------

        "file": (
            chunk_path
            .relative_to(OUTPUT_DIR)
            .as_posix()
        ),

        # --------------------------------------------
        # Existing timestamp
        # --------------------------------------------

        "created_at": CREATED_AT,
    }

    chunk_metadata.append(metadata)

    # ========================================================
    # SAVE MARKDOWN
    # ========================================================

    front_matter = {
        "chunk_id": chunk_id,
        "chunk_index": index,
        "source_file": SOURCE_FILE,

        # hierarchy
        "parent_id": parent_id,
        "parent_title": parent_title,
        "section_path": section_path,

        "created_at": CREATED_AT,

        "num_tokens": item["num_tokens"],
    }

    # YAML đơn giản
    yaml_lines = ["---"]

    for key, value in front_matter.items():

        if isinstance(value, str):

            escaped = value.replace(
                '"',
                '\\"'
            )

            yaml_lines.append(
                f'{key}: "{escaped}"'
            )

        else:

            yaml_lines.append(
                f"{key}: {value}"
            )

    yaml_lines.append("---")

    chunk_path.write_text(
        "\n".join(yaml_lines)
        + "\n\n"
        + text
        + "\n",
        encoding="utf-8",
    )

    print(
        f"[{index + 1}/{len(raw_chunks)}] "
        f"{chunk_id} "
        f"({item['num_tokens']} tokens) "
        f"parent={parent_id} "
        f"section={section_path!r} "
        f"images={len(images)} "
        f"(exact={len(exact_images)}, "
        f"bbox={len(bbox_images)}) "
        f"tables={len(tables)}"
    )


# ============================================================
# 5h. SAVE METADATA JSON
# ============================================================

(
    OUTPUT_DIR / "chunks_metadata.json"
).write_text(
    json.dumps(
        chunk_metadata,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 5i. SUMMARY
# ============================================================

print(
    f"\nSaved {len(chunk_metadata)} chunks "
    f"-> {OUTPUT_DIR / 'chunks_metadata.json'}"
)

print(
    f"Unique parents: "
    f"{len(parent_groups)}"
)

Images in map: 87 | Tables in map: 28
Pictures indexed: 87 | mapped: 82
[1/123] chunk_0000 (106 tokens) parent=parent_c94baf3ce930 section="User's Manual" images=4 (exact=0, bbox=4) tables=1
[2/123] chunk_0001 (177 tokens) parent=parent_1000ea0d2bcc section='Package Contents' images=3 (exact=0, bbox=3) tables=0
[3/123] chunk_0002 (96 tokens) parent=parent_078900f3cbec section='D The Nikon Download Center' images=1 (exact=0, bbox=1) tables=0
[4/123] chunk_0003 (67 tokens) parent=parent_5025b400c715 section='❚❚ Symbols' images=3 (exact=0, bbox=3) tables=0
[5/123] chunk_0004 (160 tokens) parent=parent_5d5897749cab section='❚❚ Conventions' images=0 (exact=0, bbox=0) tables=0
[6/123] chunk_0005 (39 tokens) parent=parent_f9f2b7f622c4 section='A For Your Safety' images=0 (exact=0, bbox=0) tables=0
[7/123] chunk_0006 (61 tokens) parent=parent_3851fd4c3ffa section='❚❚ Nikon User Support' images=0 (exact=0, bbox=0) tables=0
[8/123] chunk_0007 (512 tokens) parent=parent_3851fd4c3ffa section='❚❚ N

In [8]:
# ============================================================
# 6. QDRANT SETUP (docker compose up -d truoc khi chay cell nay)
# ============================================================
# %pip install -U qdrant-client

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

QDRANT_URL = "http://localhost:6333"
COLLECTION = "rag_docs"
VECTOR_SIZE = 2048  # RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0
RECREATE = False    # True = xoa lam lai tu dau

client = QdrantClient(url=QDRANT_URL)



In [9]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance,
)

client.create_collection(
    collection_name=COLLECTION,

    vectors_config={
        "dense": VectorParams(
            size=2048,
            distance=Distance.COSINE,
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index={
                "on_disk": False,
            }
        )
    },
)

2026-10-01 00:50:14,236 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-10-01 00:50:14,252 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"


True

In [18]:
import math
import time
import base64
from pathlib import Path
from typing import Any
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams, SparseVectorParams, SparseVector
# ============================================================
# CONFIG
# ============================================================
QDRANT_URL = "http://localhost:6333"
OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"
COLLECTION = "rag_docs"
OUTPUT_DIR = Path("./output")
CHUNKS_DIR = OUTPUT_DIR / "chunks"
IMAGE_ID_OFFSET = 100000
TEXT_BATCH = 16
VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"
# ============================================================
# CLIENTS
# ============================================================
client = QdrantClient(url=QDRANT_URL)
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_URL,
)
sparse_embeddings = FastEmbedSparse(
    model_name="Qdrant/bm25"
)
# ============================================================
# LOAD METADATA
# ============================================================
chunks_meta = json.loads(
    (OUTPUT_DIR / "chunks_metadata.json").read_text(encoding="utf-8")
)
images_meta = json.loads(
    (OUTPUT_DIR / "images_metadata.json").read_text(encoding="utf-8")
)
tables_meta = json.loads(
    (OUTPUT_DIR / "tables_metadata.json").read_text(encoding="utf-8")
)
# ============================================================
# LOAD TABLE CSV
# ============================================================
table_csv = {}
for table in tables_meta:
    if table.get("csv"):
        path = OUTPUT_DIR / table["csv"]
        if path.exists():
            table_csv[table["table_id"]] = path.read_text(
                encoding="utf-8-sig"
            )
# ============================================================
# HELPERS
# ============================================================
def chunk_body(chunk_id: str) -> str:
    path = CHUNKS_DIR / f"{chunk_id}.md"
    raw = path.read_text(encoding="utf-8")
    parts = raw.split("---\n")
    if len(parts) >= 3:
        return "---\n".join(parts[2:]).strip()
    return raw.strip()
def cosine(a: list[float], b: list[float]) -> float:
    denominator = math.sqrt(
        sum(x * x for x in a) * sum(x * x for x in b)
    )
    if denominator == 0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / denominator
def get_existing_ids() -> set:
    existing = set()
    offset = None
    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION,
            limit=512,
            offset=offset,
            with_vectors=False,
            with_payload=False,
        )
        existing.update(point.id for point in points)
        if offset is None:
            break
    return existing
# ============================================================
# CREATE HYBRID COLLECTION
# ============================================================
if not client.collection_exists(COLLECTION):
    dense_dimension = len(
        embeddings.embed_query("dimension test")
    )
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config={
            VECTOR_NAME: VectorParams(
                size=dense_dimension,
                distance=Distance.COSINE,
            )
        },
        sparse_vectors_config={
            SPARSE_VECTOR_NAME: SparseVectorParams()
        },
    )
    print(
        f"Created collection '{COLLECTION}' "
        f"with dense={dense_dimension} + sparse=BM25"
    )
else:
    info = client.get_collection(COLLECTION)
    print("Collection already exists:")
    print("Dense vectors:", info.config.params.vectors)
    print("Sparse vectors:", info.config.params.sparse_vectors)
# ============================================================
# BUILD VECTOR STORE
# ============================================================
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name=VECTOR_NAME,
    sparse_vector_name=SPARSE_VECTOR_NAME,
)
print("retrieval_mode:", vector_store.retrieval_mode)
print("vector_name:", repr(vector_store.vector_name))
print("sparse_vector_name:", repr(vector_store.sparse_vector_name))
# ============================================================
# BUILD IMAGE -> CHUNK MAPPING
# ============================================================
img_to_chunks = {}
for chunk in chunks_meta:
    for image in chunk.get("images", []):
        image_id = image["image_id"]
        img_to_chunks.setdefault(image_id, []).append(
            chunk["chunk_id"]
        )
# ============================================================
# GET EXISTING POINTS
# ============================================================
existing = get_existing_ids()
print(
    f"Existing points in Qdrant: {len(existing)}"
)
# ============================================================
# CHUNK INGEST
# ============================================================
todo = [
    i
    for i in range(len(chunks_meta))
    if i not in existing
]
print(
    f"Chunks to embed: "
    f"{len(todo)}/{len(chunks_meta)}"
)

if todo:
    bodies = []
    inputs = []
    for i in todo:
        chunk = chunks_meta[i]
        body = chunk_body(chunk["chunk_id"])
        bodies.append(body)
        extra = ""
        for table in chunk.get("tables", []):
            table_id = table["table_id"]
            if table_id in table_csv:
                extra += (
                    f"\n\n[TABLE {table_id} "
                    f"page {table.get('page_no')}]\n"
                    f"{table_csv[table_id]}"
                )
        inputs.append(body + extra)
    dense_vectors = []
    total_batches = (
        len(inputs) + TEXT_BATCH - 1
    ) // TEXT_BATCH
    for start in range(
        0,
        len(inputs),
        TEXT_BATCH,
    ):
        batch = inputs[start:start + TEXT_BATCH]
        vectors = embeddings.embed_documents(batch)
        dense_vectors.extend(vectors)
        batch_no = start // TEXT_BATCH + 1
        print(
            f"Dense batch "
            f"{batch_no}/{total_batches}"
        )
    print("Generating sparse BM25 vectors...")
    sparse_vectors = sparse_embeddings.embed_documents(
        inputs
    )
    points = []
    for i, body, dense_vector, sparse_vector in zip(
        todo,
        bodies,
        dense_vectors,
        sparse_vectors,
    ):
        chunk = chunks_meta[i]
        pages = sorted(
            {
                item.get("page_no")
                for item in (
                    chunk.get("images", [])
                    + chunk.get("tables", [])
                )
                if item.get("page_no") is not None
            }
        )
        sparse_vector = SparseVector(
            indices=list(sparse_vector.indices),
            values=list(sparse_vector.values),
        )
        payload = {
    "page_content": body,
    "metadata": {
        "type": "chunk",

        # Chunk
        "chunk_id": chunk["chunk_id"],
        "chunk_index": chunk["chunk_index"],

        # Parent / hierarchy
        "parent_id": chunk.get("parent_id"),
        "parent_title": chunk.get("parent_title"),
        "section_path": chunk.get("section_path"),

        # Document
        "source_file": chunk["source_file"],
        "num_tokens": chunk.get("num_tokens"),
        "page_nos": pages,

        # Assets
        "images": chunk.get("images", []),
        "tables": chunk.get("tables", []),

        # Time
        "created_at": chunk.get("created_at"),
        "updated_at": chunk.get("updated_at"),

        # File
        "file": chunk.get("file"),
    },
}
        points.append(
            PointStruct(
                id=i,
                vector={
                    VECTOR_NAME: dense_vector,
                    SPARSE_VECTOR_NAME: sparse_vector,
                },
                payload=payload,
            )
        )
    client.upsert(
        collection_name=COLLECTION,
        points=points,
    )
    print(
        f"Upserted {len(points)} hybrid chunk points"
    )
# ============================================================
# VISION CHECK
# ============================================================
def vision_works() -> bool:
    if len(images_meta) < 2:
        print(
            "Not enough images for vision check."
        )
        return False
    try:
        image_vectors = []
        for image_meta in images_meta[:2]:
            image_path = (
                OUTPUT_DIR / image_meta["file"]
            )
            image_b64 = base64.b64encode(
                image_path.read_bytes()
            ).decode("utf-8")
            vector = embeddings.embed_query(
                image_b64
            )
            image_vectors.append(vector)
        similarity = cosine(
            image_vectors[0],
            image_vectors[1],
        )
        print(
            "Vision check cosine:",
            f"{similarity:.6f}",
        )
        return similarity < 0.999
    except Exception as e:
        print(
            "Vision check failed:",
            repr(e),
        )
        return False
# ============================================================
# IMAGE INGEST
# ============================================================
if vision_works():
    todo_images = [
        (index, image)
        for index, image in enumerate(images_meta)
        if IMAGE_ID_OFFSET + index not in existing
    ]
    print(
        f"Images to embed: "
        f"{len(todo_images)}/{len(images_meta)}"
    )
    for position, (index, image) in enumerate(
        todo_images,
        start=1,
    ):
        image_path = OUTPUT_DIR / image["file"]
        image_b64 = base64.b64encode(
            image_path.read_bytes()
        ).decode("utf-8")
        # Nếu model của bạn hỗ trợ image input qua Ollama,
        # phần image embedding nên gọi Ollama trực tiếp.
        # LangChain OllamaEmbeddings hiện tại không phải
        # interface phù hợp cho multimodal image input.
        print(
            f"[{position}/{len(todo_images)}] "
            f"Skipping image point: "
            f"custom multimodal embedding required -> "
            f"{image['image_id']}"
        )
else:
    print(
        "WARNING: vision embedding is unavailable."
    )
# ============================================================
# CHECK RESULT
# ============================================================
info = client.get_collection(COLLECTION)
print(
    "points:",
    info.points_count,
)
print(
    "dense:",
    info.config.params.vectors,
)
print(
    "sparse:",
    info.config.params.sparse_vectors,
)

2026-10-01 00:58:27,040 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-10-01 00:58:27,535 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25 "HTTP/1.1 200 OK"
2026-10-01 00:58:27,789 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25/tree/22b8d2af71a76161e18dd432d2cee0eefa66e412?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-10-01 00:58:28,034 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25/revision/main "HTTP/1.1 200 OK"
2026-10-01 00:58:28,292 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs/exists "HTTP/1.1 200 OK"
2026-10-01 00:58:36,087 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 00:58:36,227 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"
2026-10-01 00:58:36,233 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"
2026-10-01 00:58:36,286 - INF

Created collection 'rag_docs' with dense=2048 + sparse=BM25
retrieval_mode: RetrievalMode.HYBRID
vector_name: 'dense'
sparse_vector_name: 'sparse'
Existing points in Qdrant: 0
Chunks to embed: 123/123


2026-10-01 00:58:52,480 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Dense batch 1/8


2026-10-01 00:59:03,699 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Dense batch 2/8


... (12 outputs left out)

2026-10-01 01:00:12,640 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/points?wait=true "HTTP/1.1 200 OK"


Upserted 123 hybrid chunk points


2026-10-01 01:00:28,352 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 01:00:43,716 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 01:00:43,784 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"


Vision check cosine: 0.975594
Images to embed: 87/87
[1/87] Skipping image point: custom multimodal embedding required -> picture_0001
[2/87] Skipping image point: custom multimodal embedding required -> picture_0002
[3/87] Skipping image point: custom multimodal embedding required -> picture_0003
[4/87] Skipping image point: custom multimodal embedding required -> picture_0004
[5/87] Skipping image point: custom multimodal embedding required -> picture_0005
[6/87] Skipping image point: custom multimodal embedding required -> picture_0006
[7/87] Skipping image point: custom multimodal embedding required -> picture_0007
[8/87] Skipping image point: custom multimodal embedding required -> picture_0008
[9/87] Skipping image point: custom multimodal embedding required -> picture_0009
[10/87] Skipping image point: custom multimodal embedding required -> picture_0010
[11/87] Skipping image point: custom multimodal embedding required -> picture_0011
[12/87] Skipping image point: custom multim

In [19]:
OLLAMA_URL_CHAT = "http://localhost:11434/api/chat"
model = "qwen3:1.7b"

In [20]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=model,
    temperature=0,
)

In [21]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

2026-10-01 01:02:04,639 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


AIMessage(content='Je aime la programmation.', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-09-30T18:02:10.1064296Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13096570100, 'load_duration': 7261454600, 'prompt_eval_count': 35, 'prompt_eval_duration': 316702000, 'eval_count': 288, 'eval_duration': 5503847000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a0f37a-f5ea-70d1-b1b5-ad0a5455d2e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 288, 'total_tokens': 323})

In [22]:
from langchain_ollama import OllamaEmbeddings
EMBED_MODEL = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL
)

In [23]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import (
    QdrantVectorStore,
    RetrievalMode,
)
from langchain_qdrant import FastEmbedSparse
sparse_embeddings = FastEmbedSparse(
    model_name="Qdrant/bm25"
)
QDRANT_URL = "http://localhost:6333"
client = QdrantClient(QDRANT_URL)



vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION,

    # Dense
    embedding=embeddings,

    # Sparse
    sparse_embedding=sparse_embeddings,

    # Hybrid
    retrieval_mode=RetrievalMode.HYBRID,

    # Tên 2 vector trong Qdrant
    vector_name="dense",
    sparse_vector_name="sparse",
    content_payload_key="page_content",
    metadata_payload_key="metadata",
)

2026-10-01 01:02:40,462 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25 "HTTP/1.1 200 OK"
2026-10-01 01:02:40,709 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25/tree/22b8d2af71a76161e18dd432d2cee0eefa66e412?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-10-01 01:02:40,968 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qdrant/bm25/revision/main "HTTP/1.1 200 OK"
2026-10-01 01:02:41,403 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"
2026-10-01 01:02:41,581 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-10-01 01:02:48,515 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 01:02:48,521 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"


In [24]:
info = client.get_collection(COLLECTION)

print(info)
print(info.config.params.vectors)
print(info.config.params.sparse_vectors)

2026-10-01 01:02:52,726 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=123 points_count=123 segments_count=6 config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=2048, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors={'sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshol

In [25]:
print("retrieval_mode =", vector_store.retrieval_mode)
print("vector_name =", repr(vector_store.vector_name))
print("sparse_vector_name =", repr(vector_store.sparse_vector_name))

retrieval_mode = RetrievalMode.HYBRID
vector_name = 'dense'
sparse_vector_name = 'sparse'


In [29]:
from datetime import datetime, timezone
from qdrant_client import models

now = datetime.now(timezone.utc)

qdrant_filter = models.Filter(
    must=[
        # source_file nằm trong metadata
        models.FieldCondition(
            key="metadata.source_file",
            match=models.MatchValue(
                value="Z5UM_EU(En)04.pdf"
            )
        ),

        # updated_at nằm ở ROOT
        # models.FieldCondition(
        #     key="updated_at",
        #     range=models.DatetimeRange(
        #         gte="2026-09-28T08:59:10.917966+00:00",
        #         lte=now,
        #     )
        # ),
    ]
)

ans = vector_store.similarity_search_with_relevance_scores(
    query="How do I create or customize a Picture Control on the Nikon Z5?",
    k=5,
    filter=qdrant_filter,
)

2026-10-01 01:03:12,150 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 01:03:12,201 - INFO - HTTP Request: POST http://localhost:6333/collections/rag_docs/points/query "HTTP/1.1 200 OK"


In [30]:
ans

[(Document(metadata={'type': 'chunk', 'chunk_id': 'chunk_0071', 'chunk_index': 71, 'parent_id': 'parent_64db013fd03f', 'parent_title': 'Nikon Z 5 Digital Camera', 'section_path': 'Nikon Z 5 Digital Camera', 'source_file': 'Z5UM_EU(En)04.pdf', 'num_tokens': 320, 'page_nos': [44], 'images': [], 'tables': [{'table_id': 'table_0012', 'csv': 'tables/table_0012.csv', 'html': 'tables/table_0012.html', 'page_no': 44, 'doc_item_ref': '#/tables/11'}], 'created_at': '2026-10-01T00:46:58+07:00', 'updated_at': None, 'file': 'chunks/chunk_0071.md', '_id': 71, '_collection_name': 'rag_docs'}, page_content='Nikon Z 5 Digital Camera\nPicture Control System,  = Auto, Standard, Neutral, Vivid, Monochrome, Portrait, Landscape, Flat, Creative Picture Controls (Dream, Morning, Pop, Sunday, Somber, Dramatic, Silence, Bleached, Melancholic, Pure, Denim, Toy, Sepia, Blue, Red, Pink, Charcoal, Graphite, Binary, Carbon); selected Picture Control can be modified; storage for custom Picture Controls. Media,  = SD 

In [60]:
ans[0][0].page_content

'Nikon Z 5 Digital Camera\nPicture Control System,  = Auto, Standard, Neutral, Vivid, Monochrome, Portrait, Landscape, Flat, Creative Picture Controls (Dream, Morning, Pop, Sunday, Somber, Dramatic, Silence, Bleached, Melancholic, Pure, Denim, Toy, Sepia, Blue, Red, Pink, Charcoal, Graphite, Binary, Carbon); selected Picture Control can be modified; storage for custom Picture Controls. Media,  = SD (Secure Digital) and UHS-II compliant SDHC and SDXC memory cards. Double card slots,  = The card in Slot 2 can be used for overflow or backup storage or for separate storage of NEF (RAW) and JPEG images; pictures can be copied between cards.. File system,  = DCF 2.0, Exif 2.31. Viewfinder,  = . Viewfinder,  = 1.27-cm/0.5-in. approx. 3690k-dot (Quad VGA) OLED electronic viewfinder with color balance and auto and 11-level manual brightness controls. Frame coverage,  = Approx. 100% horizontal and 100% vertical. Magnification,  = Approx. 0.8× (50 mmlens at infinity, -1.0m -1 ). Eyepoint,  = 21 m

In [32]:
query = "How do I create or customize a Picture Control on the Nikon Z5?"

ans = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=20,
    hybrid_fusion=models.FusionQuery(
        fusion=models.Fusion.DBSF
    )
)

print(len(ans))

for doc, score in ans:
    print(
        f"{score:.4f}",
        doc.metadata["chunk_id"],
        doc.metadata.get("parent_title"),
        doc.metadata.get("page_nos")
    )

2026-10-01 01:07:07,256 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-10-01 01:07:07,286 - INFO - HTTP Request: POST http://localhost:6333/collections/rag_docs/points/query "HTTP/1.1 200 OK"


20
1.4221 chunk_0071 Nikon Z 5 Digital Camera [44]
1.2706 chunk_0013 ❚❚ Nikon User Support [7, 8]
1.1704 chunk_0014 ❚❚ Nikon User Support [9]
1.0741 chunk_0077 Nikon Z 5 Digital Camera [50]
1.0041 chunk_0016 ❚❚ Nikon User Support [10]
0.9499 chunk_0015 ❚❚ Nikon User Support [10]
0.9327 chunk_0075 Nikon Z 5 Digital Camera [48]
0.8565 chunk_0073 Nikon Z 5 Digital Camera [46]
0.8467 chunk_0072 Nikon Z 5 Digital Camera [45]
0.8266 chunk_0018 ❚❚ Nikon User Support [11]
0.8181 chunk_0068 1 Press the K button. [40]
0.8178 chunk_0069 2 Press 4 or 2 to view additional pictures. [40]
0.7740 chunk_0019 e Bracketing/flash [12, 13, 14]
0.7717 chunk_0074 Nikon Z 5 Digital Camera [47]
0.7609 chunk_0037 D Before Taking Important Pictures []
0.7579 chunk_0106 Notice for Customers in the UK [66]
0.7554 chunk_0105 Notice for Customers in Europe and in Countries Complying with the Radio Equipment Directive [65]
0.7495 chunk_0121 Dear Nikon Customer, []
0.7447 chunk_0096 Bluetooth and Wi-Fi (Wireless LAN) 

C:\Users\Admin\AppData\Local\Temp\ipykernel_29312\1828079484.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'type': 'chunk', 'chunk_id': 'chunk_0071', 'chunk_index': 71, 'parent_id': 'parent_64db013fd03f', 'parent_title': 'Nikon Z 5 Digital Camera', 'section_path': 'Nikon Z 5 Digital Camera', 'source_file': 'Z5UM_EU(En)04.pdf', 'num_tokens': 320, 'page_nos': [44], 'images': [], 'tables': [{'table_id': 'table_0012', 'csv': 'tables/table_0012.csv', 'html': 'tables/table_0012.html', 'page_no': 44, 'doc_item_ref': '#/tables/11'}], 'created_at': '2026-10-01T00:46:58+07:00', 'updated_at': None, 'file': 'chunks/chunk_0071.md', '_id': 71, '_collection_name': 'rag_docs'}, page_content='Nikon Z 5 Digital Camera\nPicture Control System,  = Auto, Standard, Neutral, Vivid, Monochrome, Portrait, Landscape, Flat, Creative Picture Controls (Dream, Morning, Pop, Sunday, Somber, Dramatic, Silence, Bleached, Melancholic, Pure, Denim, Toy, Sepia, Blue, Red, Pink, Cha

In [33]:
from collections import defaultdict

parent_groups = defaultdict(list)

for doc, score in ans:
    parent_id = doc.metadata["parent_id"]

    parent_groups[parent_id].append({
        "doc": doc,
        "score": score,
    })

for parent_id, items in parent_groups.items():
    print("=" * 80)
    print("PARENT:", parent_id)
    print("TITLE:", items[0]["doc"].metadata["parent_title"])

    for item in items:
        doc = item["doc"]
        print(
            doc.metadata["chunk_id"],
            "score=",
            item["score"],
            "page=",
            doc.metadata.get("page_nos")
        )

PARENT: parent_64db013fd03f
TITLE: Nikon Z 5 Digital Camera
chunk_0071 score= 1.4220637 page= [44]
chunk_0077 score= 1.07414005 page= [50]
chunk_0075 score= 0.932733 page= [48]
chunk_0073 score= 0.8564963999999999 page= [46]
chunk_0072 score= 0.8467450000000001 page= [45]
chunk_0074 score= 0.77166283 page= [47]
PARENT: parent_3851fd4c3ffa
TITLE: ❚❚ Nikon User Support
chunk_0013 score= 1.2706165 page= [7, 8]
chunk_0014 score= 1.17036355 page= [9]
chunk_0016 score= 1.0040654500000001 page= [10]
chunk_0015 score= 0.94990057 page= [10]
chunk_0018 score= 0.8265572800000001 page= [11]
chunk_0017 score= 0.738843515 page= [11]
PARENT: parent_1129c3fe5ba0
TITLE: 1 Press the K button.
chunk_0068 score= 0.8181087 page= [40]
PARENT: parent_88e15b0fca02
TITLE: 2 Press 4 or 2 to view additional pictures.
chunk_0069 score= 0.817819325 page= [40]
PARENT: parent_d70d9e2a087f
TITLE: e Bracketing/flash
chunk_0019 score= 0.77398655 page= [12, 13, 14]
PARENT: parent_9f1e1523c4a6
TITLE: D Before Taking Impo

In [38]:
from sentence_transformers import CrossEncoder

# Load reranker
reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B"
)

docs = [doc for doc, _ in ans]

pairs = [
    (query, doc.page_content)
    for doc in docs
]

rerank_scores = reranker.predict(pairs)

reranked = sorted(
    zip(docs, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

for rank, (doc, score) in enumerate(reranked[:8], 1):
    print(
        rank,
        f"{score:.4f}",
        doc.metadata["chunk_id"],
        doc.metadata["parent_title"],
        doc.metadata.get("page_nos")
    )

2026-10-01 01:11:00,336 - INFO - No device provided, using cpu
2026-10-01 01:11:01,021 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Reranker-0.6B/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 01:11:01,031 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Reranker-0.6B/e61197ed45024b0ed8a2d74b80b4d909f1255473/modules.json?%2FQwen%2FQwen3-Reranker-0.6B%2Fresolve%2Fmain%2Fmodules.json=&etag=%22008ceadb8aca5de2810344509a4ac73352f23428%22 "HTTP/1.1 200 OK"
2026-10-01 01:11:01,285 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Reranker-0.6B/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 01:11:01,295 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Reranker-0.6B/e61197ed45024b0ed8a2d74b80b4d909f1255473/config_sentence_transformers.json?%2FQwen%2FQwen3-Reranker-0.6B%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

2026-10-01 01:11:05,564 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Reranker-0.6B "HTTP/1.1 200 OK"
2026-10-01 01:11:05,829 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Reranker-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-01 01:11:06,084 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Reranker-0.6B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-01 01:11:08,409 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Reranker-0.6B "HTTP/1.1 200 OK"
2026-10-01 01:11:08,664 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Reranker-0.6B/resolve/main/1_LogitScore/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 01:11:08,673 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Reranker-0.6B/e61197ed45024b0ed8a2d74b80b4d909f1255473/1_LogitScore%2Fconfig.json?%2FQwen%2FQwe

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1 7.7500 chunk_0071 Nikon Z 5 Digital Camera [44]
2 2.0625 chunk_0013 ❚❚ Nikon User Support [7, 8]
3 1.4375 chunk_0072 Nikon Z 5 Digital Camera [45]
4 1.3125 chunk_0077 Nikon Z 5 Digital Camera [50]
5 0.7500 chunk_0014 ❚❚ Nikon User Support [9]
6 -0.0625 chunk_0075 Nikon Z 5 Digital Camera [48]
7 -0.3125 chunk_0073 Nikon Z 5 Digital Camera [46]
8 -0.5000 chunk_0074 Nikon Z 5 Digital Camera [47]


In [41]:
top_docs = [
    doc
    for doc, score in reranked[:5]
]

In [42]:
from collections import Counter

parent_counts = Counter(
    doc.metadata["parent_id"]
    for doc in top_docs
)

print(parent_counts)

Counter({'parent_64db013fd03f': 3, 'parent_3851fd4c3ffa': 2})


In [43]:
from collections import defaultdict

grouped = defaultdict(list)

for doc, score in reranked[:5]:
    grouped[doc.metadata["parent_id"]].append(
        (doc, score)
    )

for parent_id, items in grouped.items():
    print("\n", "=" * 80)
    print("Parent:", items[0][0].metadata["parent_title"])

    for doc, score in items:
        print(
            f"{doc.metadata['chunk_id']} "
            f"| score={score:.4f} "
            f"| pages={doc.metadata.get('page_nos')}"
        )


Parent: Nikon Z 5 Digital Camera
chunk_0071 | score=7.7500 | pages=[44]
chunk_0072 | score=1.4375 | pages=[45]
chunk_0077 | score=1.3125 | pages=[50]

Parent: ❚❚ Nikon User Support
chunk_0013 | score=2.0625 | pages=[7, 8]
chunk_0014 | score=0.7500 | pages=[9]


In [44]:
parent_scores = {}

for parent_id, items in grouped.items():
    scores = [score for _, score in items]

    parent_scores[parent_id] = {
        "max_score": max(scores),
        "mean_score": sum(scores) / len(scores),
        "count": len(scores),
    }

for parent_id, info in parent_scores.items():
    print(parent_id, info)

parent_64db013fd03f {'max_score': 7.75, 'mean_score': 3.5, 'count': 3}
parent_3851fd4c3ffa {'max_score': 2.0625, 'mean_score': 1.40625, 'count': 2}
